#### 複数のGWAS Summary Stats などを使ってMeta GWASを行うためのものNotebookである。基本的には、スクリプトを使ってやる

In [1]:
import pandas as pd

In [7]:
import pandas as pd

df = pd.read_csv(
    "/Users/yoshizawakazuki/Desktop/Snore_women_GWAS/chr13/summary_stats.gz",
    sep="\t"
)


In [8]:
df.head()

,#chrom,pos,rsid,ref,alt,neg_log_pvalue,beta,stderr_beta,alt_allele_freq
0,13,16002801,.,C,A,0.40226533455244756,1.25781,.,.
1,13,16002806,.,T,A,0.1357809199325776,1.45996,.,.
2,13,16002867,.,A,G,0.8391504653632305,0.291985,.,.
3,13,16002876,.,G,A,0.07592653556375667,1.0398,.,.
4,13,16002919,.,C,T,0.37800991926743,0.875965,.,.


## GWAS_Snore. Metaanalysis用の一連の流れを下に示す：Claude AIと実装；今後適宜変えていきたい

In [22]:
import pandas as pd

df1 = pd.read_csv(
    "/Users/yoshizawakazuki/Downloads/GCST90692131.tsv.gz",
    sep="\t"
)


In [23]:
df1.head(10)

,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,neg_log_10_p_value
0,1,11063,G,T,-0.99960,1.21600,0.000052,0.38590
1,1,13259,A,G,-0.07279,0.16390,0.000286,0.18240
2,1,17641,A,G,0.03503,0.09556,0.000807,0.14640
3,1,57222,C,T,-0.02277,0.10870,0.000672,0.07881
4,1,58396,C,T,0.06282,0.16490,0.000229,0.15290
5,1,63668,A,G,-0.25420,0.42700,0.001874,0.25830
6,1,69569,C,T,0.14250,0.19310,0.000179,0.33670
7,1,79192,G,T,0.25620,0.29280,0.000073,0.41840
8,1,91588,A,G,0.07817,0.19810,0.000159,0.15910
9,1,533573,A,G,-0.01849,0.10230,0.000681,0.06721


In [11]:
df2 = pd.read_csv(
    "/Users/yoshizawakazuki/Downloads/Campos_prePMID_Snoring-mainAnalysis.gz",
    sep="\t"
)

In [13]:
df2.head(10)

,CHR,BP,SNP,A1,A2,FREQ,BETA,SE,P
0,1,636285,rs545945172,T,C,0.904303,0.002157,0.002070,0.31
1,1,649192,rs201942322,A,T,0.883020,0.002092,0.001870,0.28
2,1,662414,rs371628865,C,T,0.865541,0.002332,0.001833,0.21
3,1,662622,rs61769339,G,A,0.890009,0.002171,0.001876,0.26
4,1,665266,rs539032812,T,C,0.981032,0.005793,0.004654,0.21
5,1,692794,1:692794_CA_C,CA,C,0.889564,0.002931,0.001836,0.11
6,1,693731,rs12238997,A,G,0.884242,0.002353,0.001733,0.18
7,1,693823,rs61769351,G,C,0.888548,0.002325,0.001867,0.23
8,1,701835,rs189800799,T,C,0.968334,0.002554,0.003606,0.48
9,1,705882,rs72631875,G,A,0.933106,0.002279,0.002541,0.36


In [16]:
df3 = pd.read_csv("/Users/yoshizawakazuki/Downloads/GCST90691587.tsv.gz",sep="\t")
df3.head(20)

,chromosome,base_pair_location,effect_allele,other_allele,beta,standard_error,effect_allele_frequency,neg_log_10_p_value
0,1,11063,G,T,-0.99960,1.21600,0.000052,0.38590
1,1,13259,A,G,-0.07279,0.16390,0.000286,0.18240
2,1,17641,A,G,0.03503,0.09556,0.000807,0.14640
3,1,57222,C,T,-0.02277,0.10870,0.000672,0.07881
4,1,58396,C,T,0.06282,0.16490,0.000229,0.15290
5,1,63668,A,G,-0.25420,0.42700,0.001874,0.25830
6,1,69569,C,T,0.14250,0.19310,0.000179,0.33670
7,1,79192,G,T,0.25620,0.29280,0.000073,0.41840
8,1,91588,A,G,0.07817,0.19810,0.000159,0.15910
9,1,533573,A,G,-0.01849,0.10230,0.000681,0.06721


In [21]:
len(df3),len(df),len(df2) # df とdf2の間でMeta Analysisを行うか

(23406520, 23406520, 11010158)

In [15]:
df.columns,df2.columns,df3.columns

(Index(['chromosome', 'base_pair_location', 'effect_allele', 'other_allele',
        'beta', 'standard_error', 'effect_allele_frequency',
        'neg_log_10_p_value'],
       dtype='object'),
 Index(['CHR', 'BP', 'SNP', 'A1', 'A2', 'FREQ', 'BETA', 'SE', 'P'], dtype='object'),
 Index(['chromosome', 'base_pair_location', 'effect_allele', 'other_allele',
        'beta', 'standard_error', 'effect_allele_frequency',
        'neg_log_10_p_value'],
       dtype='object'))

In [25]:
import pandas as pd
import numpy as np

# データの読み込み（例）
# df1: chromosome, base_pair_location, effect_allele, other_allele, beta, standard_error, effect_allele_frequency, neg_log_10_p_value
# df2: CHR, BP, SNP, A1, A2, FREQ, BETA, SE, P

# ========================================
# ステップ1: カラム名を統一
# ========================================

def standardize_format1(df):
    """Format 1 を標準形式に変換"""
    df_std = df.copy()
    
    df_std = df_std.rename(columns={
        'chromosome': 'CHR',
        'base_pair_location': 'BP',
        'effect_allele': 'A1',
        'other_allele': 'A2',
        'beta': 'BETA',
        'standard_error': 'SE',
        'effect_allele_frequency': 'FREQ',
        'neg_log_10_p_value': 'NEG_LOG10_P'
    })
    
    # P値を計算
    df_std['P'] = 10 ** (-df_std['NEG_LOG10_P'])
    
    # SNP IDを作成（chr:pos形式）
    df_std['SNP'] = 'chr' + df_std['CHR'].astype(str) + ':' + df_std['BP'].astype(str)
    
    return df_std

def standardize_format2(df):
    """Format 2 は既に標準形式なのでそのまま"""
    df_std = df.copy()
    
    # SNPカラムがない場合は作成
    if 'SNP' not in df_std.columns or df_std['SNP'].isna().any():
        df_std['SNP'] = 'chr' + df_std['CHR'].astype(str) + ':' + df_std['BP'].astype(str)
    
    return df_std

# データを変換
df1_std = standardize_format1(df1)
df2_std = standardize_format2(df2)

print("Dataset 1 standardized:")
print(df1_std.head())
print(f"Shape: {df1_std.shape}\n")

print("Dataset 2 standardized:")
print(df2_std.head())
print(f"Shape: {df2_std.shape}\n")



Dataset 1 standardized:
   CHR     BP A1 A2     BETA       SE      FREQ  NEG_LOG10_P         P  \
0    1  11063  G  T -0.99960  1.21600  0.000052      0.38590  0.411244   
1    1  13259  A  G -0.07279  0.16390  0.000286      0.18240  0.657052   
2    1  17641  A  G  0.03503  0.09556  0.000807      0.14640  0.713839   
3    1  57222  C  T -0.02277  0.10870  0.000672      0.07881  0.834046   
4    1  58396  C  T  0.06282  0.16490  0.000229      0.15290  0.703234   

          SNP  
0  chr1:11063  
1  chr1:13259  
2  chr1:17641  
3  chr1:57222  
4  chr1:58396  
Shape: (23406520, 10)

Dataset 2 standardized:
   CHR      BP          SNP A1 A2      FREQ      BETA        SE     P
0    1  636285  rs545945172  T  C  0.904303  0.002157  0.002070  0.31
1    1  649192  rs201942322  A  T  0.883020  0.002092  0.001870  0.28
2    1  662414  rs371628865  C  T  0.865541  0.002332  0.001833  0.21
3    1  662622   rs61769339  G  A  0.890009  0.002171  0.001876  0.26
4    1  665266  rs539032812  T  C  0.9

In [26]:
# ========================================
# ステップ2: データクリーニング
# ========================================

def clean_gwas_data(df, study_name):
    """GWASデータのクリーニング"""
    df_clean = df.copy()
    
    initial_count = len(df_clean)
    
    # 欠損値を削除
    df_clean = df_clean.dropna(subset=['CHR', 'BP', 'A1', 'A2', 'BETA', 'SE', 'P', 'FREQ'])
    print(f"{study_name}: Removed {initial_count - len(df_clean)} rows with missing values")
    
    # P値の範囲チェック
    df_clean = df_clean[(df_clean['P'] > 0) & (df_clean['P'] <= 1)]
    
    # SEが正の値かチェック
    df_clean = df_clean[df_clean['SE'] > 0]
    
    # 頻度が0-1の範囲内かチェック
    df_clean = df_clean[(df_clean['FREQ'] >= 0) & (df_clean['FREQ'] <= 1)]
    
    # アレルが同じでないことをチェック
    df_clean = df_clean[df_clean['A1'] != df_clean['A2']]
    
    # 染色体フィルター（1-22のみ、または1-22,X,Y,MT）
    valid_chrs = [str(i) for i in range(1, 23)]
    df_clean['CHR'] = df_clean['CHR'].astype(str)
    df_clean = df_clean[df_clean['CHR'].isin(valid_chrs)]
    
    # 重複SNPを削除（最もP値が小さいものを保持）
    df_clean = df_clean.sort_values('P').drop_duplicates(subset=['SNP'], keep='first')
    
    print(f"{study_name}: Final count = {len(df_clean)} SNPs")
    print(f"{study_name}: Removed {initial_count - len(df_clean)} total rows\n")
    
    return df_clean

df1_clean = clean_gwas_data(df1_std, "Study 1")
df2_clean = clean_gwas_data(df2_std, "Study 2")



Study 1: Removed 0 rows with missing values
Study 1: Final count = 22597338 SNPs
Study 1: Removed 809182 total rows

Study 2: Removed 0 rows with missing values
Study 2: Final count = 10997325 SNPs
Study 2: Removed 12833 total rows



In [27]:
def harmonize_alleles(df1, df2):
    """
    2つの研究間でアレルの方向性を統一
    df1を基準として、df2を調整
    """
    # コピーを作成
    df1_work = df1.copy()
    df2_work = df2.copy()
    
    # マージキーを作成
    df1_work['merge_key'] = df1_work['CHR'].astype(str) + ':' + df1_work['BP'].astype(str)
    df2_work['merge_key'] = df2_work['CHR'].astype(str) + ':' + df2_work['BP'].astype(str)
    
    # 共通SNPのみを保持
    common_keys = set(df1_work['merge_key']) & set(df2_work['merge_key'])
    print(f"Common SNPs between studies: {len(common_keys)}")
    
    df1_common = df1_work[df1_work['merge_key'].isin(common_keys)].copy()
    df2_common = df2_work[df2_work['merge_key'].isin(common_keys)].copy()
    
    # df1を基準としてdf2をマージ
    merged = df2_common.merge(
        df1_common[['merge_key', 'A1', 'A2']],
        on='merge_key',
        suffixes=('', '_ref'),
        how='inner'
    )
    
    # アレルが一致するケース
    match_mask = (merged['A1'] == merged['A1_ref']) & (merged['A2'] == merged['A2_ref'])
    
    # アレルが逆のケース（A1とA2が入れ替わっている）
    flip_mask = (merged['A1'] == merged['A2_ref']) & (merged['A2'] == merged['A1_ref'])
    
    # 逆の場合、BETAの符号を反転、FREQを1-FREQに
    merged.loc[flip_mask, 'BETA'] = -merged.loc[flip_mask, 'BETA']
    merged.loc[flip_mask, 'FREQ'] = 1 - merged.loc[flip_mask, 'FREQ']
    
    # アレルを入れ替え
    temp_a1 = merged.loc[flip_mask, 'A1'].copy()
    merged.loc[flip_mask, 'A1'] = merged.loc[flip_mask, 'A2']
    merged.loc[flip_mask, 'A2'] = temp_a1
    
    # アレルが一致または反転したケースのみ保持
    keep_mask = match_mask | flip_mask
    merged_harmonized = merged[keep_mask].copy()
    
    print(f"After harmonization:")
    print(f"  - Matching alleles: {match_mask.sum()}")
    print(f"  - Flipped alleles: {flip_mask.sum()}")
    print(f"  - Excluded (ambiguous): {(~keep_mask).sum()}")
    print(f"  - Final SNPs: {len(merged_harmonized)}\n")
    
    # df1側も共通SNPのみに絞る
    df1_final = df1_common[df1_common['merge_key'].isin(merged_harmonized['merge_key'])].copy()
    
    # 不要なカラムを削除
    df1_final = df1_final.drop(columns=['merge_key'])
    merged_harmonized = merged_harmonized.drop(columns=['A1_ref', 'A2_ref', 'merge_key'])
    
    return df1_final, merged_harmonized

# 実行
df1_harm, df2_harm = harmonize_alleles(df1_clean, df2_clean)

print("Harmonization completed!")
print(f"Study 1: {len(df1_harm)} SNPs")
print(f"Study 2: {len(df2_harm)} SNPs")

# データの確認
print("\nStudy 1 (first 5 rows):")
print(df1_harm.head())

print("\nStudy 2 (first 5 rows):")
print(df2_harm.head())

Common SNPs between studies: 10572430
After harmonization:
  - Matching alleles: 108
  - Flipped alleles: 10560789
  - Excluded (ambiguous): 19573
  - Final SNPs: 10560897

Harmonization completed!
Study 1: 10560897 SNPs
Study 2: 10560897 SNPs

Study 1 (first 5 rows):
         CHR        BP A1  A2     BETA        SE    FREQ  NEG_LOG10_P  \
17242533  13  51340315  G   A -0.04309  0.005074  0.4547        16.70   
16433231  12  65791463  T   C -0.04199  0.005242  0.3678        14.94   
16433407  12  65814117  A  AT -0.04198  0.005248  0.3673        14.90   
16433259  12  65795603  C   T  0.04188  0.005241  0.6322        14.87   
16433217  12  65789204  T   C -0.04184  0.005241  0.3679        14.85   

                     P             SNP  
17242533  1.995262e-17  chr13:51340315  
16433231  1.148154e-15  chr12:65791463  
16433407  1.258925e-15  chr12:65814117  
16433259  1.348963e-15  chr12:65795603  
16433217  1.412538e-15  chr12:65789204  

Study 2 (first 5 rows):
  CHR        BP      

In [28]:
# 必要なカラムのみ選択
cols_needed = ['SNP', 'CHR', 'BP', 'A1', 'A2', 'FREQ', 'BETA', 'SE', 'P']

df1_metal = df1_harm[cols_needed].copy()
df2_metal = df2_harm[cols_needed].copy()

# タブ区切りで保存
df1_metal.to_csv('study1_harmonized.txt', sep='\t', index=False)
df2_metal.to_csv('study2_harmonized.txt', sep='\t', index=False)

print("Files saved:")
print(f"  - study1_harmonized.txt ({len(df1_metal)} SNPs)")
print(f"  - study2_harmonized.txt ({len(df2_metal)} SNPs)")


Files saved:
  - study1_harmonized.txt (10560897 SNPs)
  - study2_harmonized.txt (10560897 SNPs)


# Metalを走らせた後の解析

In [31]:
# 結果を読み込み
results = pd.read_csv('/Users/yoshizawakazuki/Desktop/snoring_metal/meta_results1.tbl', sep='\t')

print("=== Meta-analysis Results ===")
print(f"Total SNPs: {len(results)}")
print(f"Genome-wide significant (p < 5e-8): {(results['P-value'] < 5e-8).sum()}")

# Top SNPs
print("\nTop 10 associations:")
top_snps = results.nsmallest(10, 'P-value')
# print(top_snps[['MarkerName', 'Allele1', 'Allele2', 'Freq1', 'Effect', 
#                'StdErr', 'P-value', 'Direction', 'HetPVal', 'HetISq']])

# # 異質性のチェック
# high_het = results[results['HetPVal'] < 0.05]
# print(f"\nSNPs with significant heterogeneity (HetPVal < 0.05): {len(high_het)}")

=== Meta-analysis Results ===
Total SNPs: 21121794
Genome-wide significant (p < 5e-8): 11263

Top 10 associations:


In [32]:
top_snps.head(10)

,MarkerName,Allele1,Allele2,Freq1,FreqSE,MinFreq,MaxFreq,Effect,StdErr,P-value,Direction
10167441,rs592333,a,g,0.5561,0.0,0.5561,0.5561,-0.0091,0.0011,6.887000e-18,?-
8834002,chr13:51340315,a,g,0.5453,0.0,0.5453,0.5453,0.0431,0.0051,2.026000e-17,+?
8146891,rs10878269,t,c,0.3643,0.0,0.3643,0.3643,0.0089,0.0011,3.539000e-16,?+
8163473,12:65814117_AT_A,a,at,0.3637,0.0,0.3637,0.3637,0.0089,0.0011,3.588000e-16,?+
4035375,rs10878271,t,c,0.3643,0.0,0.3643,0.3643,0.0089,0.0011,3.632000e-16,?+
18413539,rs12368809,t,c,0.3644,0.0,0.3644,0.3644,0.0088,0.0011,4.395000e-16,?+
57140,rs1389799,a,g,0.6356,0.0,0.6356,0.6356,-0.0088,0.0011,4.477000e-16,?-
15236001,rs2336713,t,g,0.6356,0.0,0.6356,0.6356,-0.0088,0.0011,4.512000e-16,?-
7787544,rs2270547,t,c,0.3646,0.0,0.3646,0.3646,0.0088,0.0011,4.610000e-16,?+
16495572,rs9783497,a,g,0.3645,0.0,0.3645,0.3645,0.0088,0.0011,4.690000e-16,?+
